<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Streamforge.png" />

# StreamForge revenue pipeline 💰

You continue as a junior AI analyst at **StreamForge**. The finance team needs a monthly revenue report broken down by country. They run this report every Monday morning. Today they ask it by email. Tomorrow another team will ask the same kind of report with different filters and different groupings.

Your manager wants you to stop writing one-off scripts. She asks you to build a small **data pipeline** instead. A pipeline is a chain of small reusable steps. Each step takes a DataFrame as input and returns a DataFrame as output. When you change one step, the rest of the pipeline still works.

## Files you will use

- `StreamForge_payments.csv`:one row per payment attempt (300 rows). Columns: `user_id`, `payment_date`, `amount`, `payment_status`. Status can be `success`, `failed`, `pending`, or `cancelled`.
- `StreamForge_user_profiles.csv`: one row per user (50 rows). Includes the `country` column you will join.

<Note type="important">

Download the two CSV files from the resources section before you start. Place them in the same folder as this notebook.

</Note>

## Setup

Run the cell below to import Pandas and load the two source files. Look at the first rows of each file. Confirm that the columns match what the file descriptions promised.

In [69]:
import pandas as pd

# Load the data
payments = pd.read_csv(r"src/StreamForge_payments.csv")
profiles = pd.read_csv(r"src/StreamForge_user_profiles.csv")

# print the shapes of the dataframes
print(payments.shape)
print(profiles.shape)


(300, 4)
(50, 5)


In [70]:
# Look at the first few rows of payments
payments.head()

,user_id,payment_date,amount,payment_status
0,user_024,2024-03-19,17.99,success
1,user_011,2024-09-11,13.99,cancelled
2,user_019,2024-11-19,17.99,success
3,user_028,2024-11-06,17.99,success
4,user_040,2024-02-03,8.99,success


In [71]:
# Look at the first few rows of users
profiles.head()

,user_id,subscription_type,user_age,country,monthly_revenue
0,user_001,premium,32.0,DE,13.99
1,user_002,standard,68.0,CA,8.99
2,user_003,premium,35.0,NaN,17.99
3,user_004,premium,56.0,JP,8.99
4,user_005,premium,21.0,DE,13.99


## Task 1: Write a step function that filters with `query()`

The first rule of the report is simple. Finance only counts revenue from successful payments. Failed, pending, and cancelled payments must drop out before any further work.

Write a function called `keep_successful_payments`. The function takes one argument: a payments DataFrame. Inside the function, use `df.query("payment_status == 'success'")` to filter. Return the filtered DataFrame.

After you write the function, call it on the `payments` DataFrame. Print the shape before and after to confirm that you removed the rows you expected.

<Note type="hint">

A step function follows a simple shape:

```python
def step_name(df):
    result = df.query("some_condition")
    return result
```

It takes a DataFrame in. It returns a DataFrame out. Nothing else.

`query()` reads boolean expressions as strings. The string can use column names directly, like `"payment_status == 'success'"` or `"amount > 10"`.

</Note>

In [72]:
print(payments.columns)

Index(['user_id', 'payment_date', 'amount', 'payment_status'], dtype='str')


In [73]:
# Define a function to filter for successful payments
def keep_successful_payments(payments:pd.DataFrame) -> pd.DataFrame:
    return payments.query("payment_status == 'success'")

# Test the function on the raw payments DataFrame
successed_payments = keep_successful_payments(payments)
print(successed_payments)

      user_id payment_date  amount payment_status
0    user_024   2024-03-19   17.99        success
2    user_019   2024-11-19   17.99        success
3    user_028   2024-11-06   17.99        success
4    user_040   2024-02-03    8.99        success
5    user_006   2024-10-28   17.99        success
..        ...          ...     ...            ...
295  user_050   2024-07-18   17.99        success
296  user_005   2024-06-05   17.99        success
297  user_047   2024-12-17   17.99        success
298  user_032   2024-04-27    8.99        success
299  user_037   2024-08-15   13.99        success

[237 rows x 4 columns]


## Task 2: Write a step function that creates a column with `assign()`

Finance wants the report grouped by month. The payments file has a `payment_date` column with full dates like `2024-03-19`. You need a `payment_month` column with values like `2024-03` so you can group on it later.

Write a function called `add_payment_month`. The function takes a DataFrame as input. Inside the function, convert `payment_date` to a Pandas `datetime`, extract the month period, and turn it into a string. Use `assign()` to attach the new column. Return the new DataFrame.

Test the function on the output of Task 1. Print the first five rows so you can see the new column.

<Note type="hint">

`assign()` adds a column without modifying the original DataFrame. The pattern looks like this:

```python
def add_column(df):
    new_df = df.assign(new_column_name=some_expression)
    return new_df
```

To turn a date string into a month string, you can build it in two steps:

```python
as_datetime = pd.to_datetime(df["payment_date"])
as_period = as_datetime.dt.to_period("M")
as_string = as_period.astype(str)
```

Then pass `as_string` as the value for the new column.

</Note>

In [74]:
# Define a function to add a payment_month column
# def add_payment_month(payments:pd.DataFrame) -> pd.DataFrame:
#     date_du_paiement 
successed_payments.info()
def add_payment_month(df : pd.DataFrame) -> pd.DataFrame:
    as_datetime = pd.to_datetime(df['payment_date'])
    as_period = as_datetime.dt.to_period(freq="M")
    as_string = as_period.astype(str)
    result = df.assign(payment_month=as_string)
    return result


# Test the function on the filtered DataFrame from Task 1
successed_payments = add_payment_month(successed_payments)
successed_payments.head()

<class 'pandas.DataFrame'>
Index: 237 entries, 0 to 299
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         237 non-null    str    
 1   payment_date    237 non-null    str    
 2   amount          237 non-null    float64
 3   payment_status  237 non-null    str    
dtypes: float64(1), str(3)
memory usage: 9.3 KB


,user_id,payment_date,amount,payment_status,payment_month
0,user_024,2024-03-19,17.99,success,2024-03
2,user_019,2024-11-19,17.99,success,2024-11
3,user_028,2024-11-06,17.99,success,2024-11
4,user_040,2024-02-03,8.99,success,2024-02
5,user_006,2024-10-28,17.99,success,2024-10


## Task 3: Write a step function that joins another table with `merge()`

The payments file does not contain the country. The country lives in the user profiles file. You need to bring it onto each payment row before you can group by country.

Write a function called `add_user_country`. The function takes two arguments: the payments DataFrame and the users DataFrame. Inside the function, select only `user_id` and `country` from the users DataFrame, then merge it onto the payments DataFrame on `user_id` with a left join. Return the merged DataFrame.

Test the function on the output of Task 2. Confirm that the new DataFrame has a `country` column.

<Note type="hint">

A step function can take more than one DataFrame, but the **first argument** must always be the DataFrame that flows through the pipeline. The other arguments come after. This rule matters in Task 5 when you chain steps with `pipe()`.

```python
def add_user_country(df, users_df):
    user_columns = users_df[["user_id", "country"]]
    merged = df.merge(user_columns, on="user_id", how="left")
    return merged
```

</Note>

In [75]:
# Define a function to add the user's country to the payments DataFrame
def add_user_country(p:pd.DataFrame, u:pd.DataFrame)-> pd.DataFrame:
    reduced_user = u[["user_id","country"]]
    return p.merge(reduced_user, on="user_id", how="left")

# Test the function on the DataFrame from Task 2
payments_with_country = add_user_country(successed_payments, profiles)

# Look at the first few rows of the final DataFrame
payments_with_country.head()

# Look at the shape and columns of the final DataFrame
print(payments_with_country.shape)

(237, 6)


## Task 4: Write a step function that aggregates with `groupby()`

Now you have one row per successful payment with a country and a month attached. Finance wants one row per country and month with two numbers: total revenue and the number of payments.

Write a function called `aggregate_revenue`. The function takes a DataFrame as input. Inside the function, group by `country` and `payment_month`. Compute two named aggregations: `total_revenue` as the sum of `amount`, and `payment_count` as the count of `amount`. Reset the index so the grouping columns become regular columns again. Return the result.

Test the function on the output of Task 3. Print the first 10 rows.

<Note type="hint">

Named aggregations let you choose the output column names directly:

```python
grouped = df.groupby(["country", "payment_month"]).agg(
    total_revenue=("amount", "sum"),
    payment_count=("amount", "count"),
)
flat = grouped.reset_index()
```

Calling `reset_index()` is important because the next step (Task 5) needs `country` and `payment_month` as regular columns, not as a MultiIndex.

</Note>

In [80]:
# Define a function to aggregate revenue by country and payment_month
def aggregate_revenue(p:pd.DataFrame)->pd.DataFrame:
    result = p.groupby(["country","payment_month"]).agg(
        total_revenue= ("amount","sum"),
        payment_count=("amount","count"),
    ).round({"total_revenue": 2})

    flat = result.reset_index()
    return flat

# Test the function on the DataFrame from Task 3
grouped = aggregate_revenue(payments_with_country)
print(grouped)

   country payment_month  total_revenue  payment_count
0       BR       2024-01           8.99              1
1       BR       2024-02          77.94              6
2       BR       2024-03          68.95              5
3       BR       2024-04          45.97              3
4       BR       2024-05         108.93              7
..     ...           ...            ...            ...
71      US       2024-08           8.99              1
72      US       2024-09          53.97              3
73      US       2024-10          13.99              1
74      US       2024-11          44.97              3
75      US       2024-12          35.98              2

[76 rows x 4 columns]


## Task 5: Chain the four steps with `pipe()`

You now have four step functions. Each one takes a DataFrame and returns a DataFrame. Pandas has a method called `pipe()` that plugs your own functions into a chain. The chain reads top to bottom like a recipe.

Build the full pipeline in a single chained statement. Start from the raw `payments` DataFrame. Pipe through each step in order. Save the final result in a variable called `revenue_by_country_month`.

Print the first 10 rows of the result. The numbers must match what you got at the end of Task 4. The chained version is shorter and easier to read, but the output is the same.

<Note type="hint">

`pipe()` calls a function and passes the DataFrame on the left as its first argument. Extra arguments come after the function name:

```python
result = (
    payments
    .pipe(keep_successful_payments)
    .pipe(add_payment_month)
    .pipe(add_user_country, users_df=users)
    .pipe(aggregate_revenue)
)
```

The outer parentheses let you put each `.pipe()` on its own line. The pipeline reads from top to bottom in execution order.

</Note>

In [77]:
# Combine all the steps into a single pipeline
result = ( payments
    .pipe(keep_successful_payments)
    .pipe(add_payment_month)
    .pipe(add_user_country, u=profiles)
    .pipe(aggregate_revenue)
)

# Look at the 10 first rows of the final aggregated DataFrame
result.head(10)


total_revenue  payment_count
country payment_month                              
BR      2024-01                 8.99              1
        2024-02                77.94              6
        2024-03                68.95              5
        2024-04                45.97              3
        2024-05               108.93              7
        2024-06               103.93              7
        2024-07                26.98              2
        2024-08                45.97              3
        2024-09                 8.99              1
        2024-10                49.97              3

## Task 6: Reshape the output with `pivot_table()`

Finance reads the report in Excel. They prefer a wide table with one row per country and one column per month. The current output has one row per country and month combination, which is harder to read.

Write a final step function called `pivot_to_monthly_view`. The function takes the aggregated DataFrame as input. Inside the function, use `pivot_table()` to set `country` as the index, `payment_month` as the columns, and `total_revenue` as the values. Use `fill_value=0` so months with no revenue show as zero instead of `NaN`. Return the pivoted DataFrame.

Add this step to the end of your pipeline from Task 5. Save the final result in a variable called `revenue_pivot`. Print the result rounded to two decimals.

<Note type="hint">

`pivot_table()` reshapes long format into wide format:

```python
wide = df.pivot_table(
    index="country",
    columns="payment_month",
    values="total_revenue",
    fill_value=0,
)
```

The `index` becomes the row labels. The `columns` become the column headers. The `values` fill the cells.

</Note>

In [78]:
# Define a function to pivot the aggregated DataFrame to a monthly view
def pivot_to_monthly_view(df:pd.DataFrame)->pd.DataFrame:
    return df.pivot_table(
    index="country",
    columns="payment_month",
    values="total_revenue",
    fill_value=0,
)

# Test the function on the aggregated DataFrame from Task 4
tcd = pivot_to_monthly_view(result)


# Look at the first few rows of the pivoted DataFrame with the total revenue for each country and month round to 2 decimal places
print(tcd.head(10))


payment_month  2024-01  2024-02  2024-03  2024-04  2024-05  2024-06  2024-07  \
country                                                                        
BR                8.99    77.94    68.95    45.97   108.93   103.93    26.98   
CA               44.97    54.96    27.98    58.96    35.96    17.99    44.96   
DE                0.00     0.00    67.96    13.99    40.97    31.98    31.98   
FR               67.96    58.96    13.99    13.99    31.98    17.98    13.99   
JP               45.96    58.96    36.97   127.91    81.93    53.95    62.96   
UK                8.99     0.00    62.96     8.99    13.99     8.99    17.99   
US               22.98    49.97    40.97    67.95    13.99    17.99    36.97   

payment_month  2024-08  2024-09  2024-10  2024-11  2024-12  
country                                                     
BR               45.97     8.99    49.97    26.98    36.97  
CA               31.98     0.00    13.99    44.97    13.99  
DE               67.94    31.98    

## Task 7: Reuse the pipeline for a different question

The marketing team sees the report. They want the same shape, but only for one subscription tier: payments of `17.99` (the premium plan). They do not want a new script. They expect you to reuse the pipeline.

Write one more step function called `keep_premium_amount`. The function takes a DataFrame as input and uses `query()` to keep only rows where `amount == 17.99`. Return the filtered DataFrame.

Build a second pipeline that uses your new step. Drop it in right after `keep_successful_payments` so the rest of the pipeline still works. Save the result in a variable called `premium_revenue_pivot`. Print the result rounded to two decimals.

This task shows the real value of pipelines. You added one new step. You did not change any of the existing steps. The pipeline still produces a clean monthly report.

<Note type="hint">

The new pipeline looks almost identical to the previous one. Only one extra `.pipe()` appears in the chain:

```python
premium_revenue_pivot = (
    payments
    .pipe(keep_successful_payments)
    .pipe(keep_premium_amount)
    .pipe(add_payment_month)
    .pipe(add_user_country, users_df=users)
    .pipe(aggregate_revenue)
    .pipe(pivot_to_monthly_view)
)
```

</Note>

In [79]:
# Define a function to filter for payments with the premium amount == 17.99
def keep_premium_amount(df:pd.DataFrame)->pd.DataFrame:
    return df.query("amount == 17.99")

# Test the function on the raw payments DataFrame
only_premium = keep_premium_amount(payments)
print(only_premium)

# Look at the first few rows of the pivoted DataFrame with the total revenue for each country and month for the premium amount round to 2 decimal places
premium_revenue_tcd = (
        payments
        .pipe(keep_successful_payments)
        .pipe(keep_premium_amount)
        .pipe(add_payment_month)
        .pipe(add_user_country, profiles)
        .pipe(aggregate_revenue)
        .pipe(pivot_to_monthly_view)
)

print(premium_revenue_tcd.head())

      user_id payment_date  amount payment_status
0    user_024   2024-03-19   17.99        success
2    user_019   2024-11-19   17.99        success
3    user_028   2024-11-06   17.99        success
5    user_006   2024-10-28   17.99        success
6    user_027   2024-02-16   17.99        success
..        ...          ...     ...            ...
290  user_019   2024-04-07   17.99        success
291  user_011   2024-07-26   17.99        success
295  user_050   2024-07-18   17.99        success
296  user_005   2024-06-05   17.99        success
297  user_047   2024-12-17   17.99        success

[104 rows x 4 columns]
payment_month  2024-01  2024-02  2024-03  2024-04  2024-05  2024-06  2024-07  \
country                                                                        
BR                0.00    17.99    17.99    17.99    71.96    71.96    17.99   
CA               35.98    17.99     0.00    35.98     0.00    17.99    17.99   
DE                0.00     0.00    53.97     0.00    17.